In [5]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Token-based Chunking

`TokenTextSplitter` measures chunk size in **tokens** (via `tiktoken`) rather than characters. Because LLM context windows and embedding limits are defined in tokens, token-based chunks map directly onto those limits — unlike character counts, which vary unpredictably per token.

We use the `cl100k_base` encoding (used by `text-embedding-3-*` and GPT-4-class models).

## 1. Load source documents

### PDF

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
pdf_docs = PyPDFLoader(str(pdf_path)).load()
print(f"PDF: loaded {len(pdf_docs)} page(s)")

PDF: loaded 12 page(s)


### HTML

In [7]:
from langchain_community.document_loaders import BSHTMLLoader

html_path = ROOT / "assets/RAG_Courses.html"
html_docs = BSHTMLLoader(str(html_path)).load()
print(f"HTML: loaded {len(html_docs)} document(s)")

HTML: loaded 1 document(s)


## 2. Token-based chunking

In [8]:
from langchain_text_splitters import TokenTextSplitter

# chunk_size / chunk_overlap are counted in TOKENS, not characters.
splitter = TokenTextSplitter(
    encoding_name="cl100k_base",   # matches text-embedding-3-* / GPT-4-class models
    chunk_size=256,                # tokens per chunk
    chunk_overlap=20,              # tokens shared between consecutive chunks
)

docs = pdf_docs + html_docs
chunks = splitter.split_documents(docs)
print(f"Split {len(docs)} documents into {len(chunks)} token-based chunks")

Split 13 documents into 53 token-based chunks


Verify the chunks really are bounded by *token* count (not characters):

In [9]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
first = chunks[0].page_content
print(f"First chunk: {len(enc.encode(first))} tokens, {len(first)} chars")
print(f"Max tokens across all chunks: {max(len(enc.encode(c.page_content)) for c in chunks)}")
print(f"\n--- first chunk ---\n{first[:400]}")

First chunk: 89 tokens, 361 chars
Max tokens across all chunks: 256

--- first chunk ---
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage.
 Author: Prabhukumar Sivamoorthy
Prabhukumarsivamoorthy@gmail.com
